# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChanderValasai/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Ranking / scoring.

Decision: which pages should editors refresh first (a prioritized queue). We need a numeric priority score so the team can pick top-K pages under limited editor capacity. This maps to a ranking task evaluated by precision@K because the business cares about the quality of the top results, not calibration across the whole distribution.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: `is_declining_label` (boolean) — computed as `trend_direction == "down"` in the pipeline.

Where it comes from: an observed outcome measured in a later window (the prepare step derives the label from future engagement/traffic). This is an observed outcome, not a hand-crafted rule, so the model predicts a real future event.

Important: `trend_direction` and `trend_pct` are derived from the same downstream measurement and must NOT be used as features (leakage).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: precision@K (use K = 50 to match the repo baseline).

Why: the product has limited editorial capacity, so we must maximize the fraction of true declining pages in the top-K the editors will act on.

What 'good' means: the committed baseline precision@50 ≈ 0.24. A defensible target is a sizable lift over the baseline (example target: precision@50 ≥ 0.50 or roughly 2–3x lift). Report both absolute precision@50 and multiplicative lift versus the hand-rule baseline, and compute metrics on the client-holdout test set so no client's pages leak between train and test. Add confidence intervals or bootstrap by client if possible.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one client-page snapshot (a single page for a single client at one snapshot time).

Example important columns:
- `client_id`, `page_id`, `snapshot_date`
- numeric features: `avg_views_28d`, `ctr_7d`, `recency`, `content_quality_signals`, etc.
- hand-rule / baseline score columns (allowed)
- `is_declining_label` (the label, computed in prepare)

Verification checks to run in the code cell below: `df.shape`, `df.head()`, `df.client_id.nunique()`, and label prevalence (`df.is_declining_label.mean()`). These confirm the unit, coverage, and base rate before training.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML: the problem combines many weak signals, client-level heterogeneity, and non-linear interactions (seasonality, content type, recency, small value signals). A single deterministic rule (if-statement) cannot adapt weights per client or capture interactions; it either misses many true declines (low recall for top-K) or floods editors with false positives. ML (even simple tree-based models or linear models with interactions) can combine signals to increase precision in the top-ranked results while still being auditable via feature importance or SHAP.

The hand-rule baseline is a strong, interpretable starting point and should be used as the comparison; ML is justified only if it meaningfully improves precision@K on held-out clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### One-paragraph frame

For FlyRank editors deciding which pages to refresh first, we will build a scored priority list from page snapshot features predicting the observed future decline label (`is_declining_label`). Measured by precision@50 on a client-holdout test set, a "good" result meaningfully beats the hand-rule baseline (target: ~2–3x lift or precision@50 ≥ ~0.50). A wrong call wastes limited editor time or misses a declining page; false positives are costly because editor capacity is limited. A plain rule isn't enough because the signals are noisy, many weak features interact non-linearly across clients and pages, and per-client heterogeneity matters. We will claim decision-support results grounded in observed outcomes only.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with the code cells unchanged (text answers are in markdown cells)
- [x] The target is an observed outcome (not learned from a rule)
- [x] The metric (precision@50) is named and computable from data
- [x] I will evaluate on a client-holdout split so no client's pages leak between train and test